# FIFA Players EDA with PySpark SQL

This notebook contains the exploratory SQL analysis for identifying patterns in young football players.

In [ ]:
from pyspark.sql import SparkSession
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
spark = SparkSession.builder \
    .appName('Analisis de FIFA Players con SQL') \
    .getOrCreate()

In [ ]:
csv_path = 'hdfs:///user/usuario/datasets/csv/male_players.csv'
df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(csv_path)
df.createOrReplaceTempView('fifa_players')
df.printSchema()

## Top Percentile Players and Attribute Summary

In [ ]:
percentile = 95
query_top_players = f"""
    SELECT *
    FROM fifa_players
    WHERE overall >= (
        SELECT approx_percentile(overall, {percentile}/100.0)
        FROM fifa_players
    )
"""
top_players_df = spark.sql(query_top_players)
top_players_df.createOrReplaceTempView('top_players')
top_players_df.select('short_name', 'overall', 'potential').show(10, truncate=False)

In [ ]:
relevant_attributes = [
    'movement_agility', 'skill_ball_control', 'power_strength',
    'attacking_crossing', 'attacking_finishing', 'attacking_heading_accuracy',
    'defending_marking_awareness', 'defending_standing_tackle',
    'defending_sliding_tackle', 'mentality_aggression',
    'mentality_interceptions', 'mentality_positioning'
]

summary_stats = pd.DataFrame()
for attribute in relevant_attributes:
    query = f"""
        SELECT
            '{attribute}' AS attribute,
            AVG({attribute}) AS avg_value,
            MIN({attribute}) AS min_value,
            MAX({attribute}) AS max_value,
            STDDEV({attribute}) AS stddev_value
        FROM fifa_players
        WHERE player_id IN (SELECT player_id FROM top_players)
    """
    stats_df = spark.sql(query).toPandas()
    summary_stats = pd.concat([summary_stats, stats_df], ignore_index=True)

summary_stats

## Age Distribution by Nationality

In [ ]:
query_age_distribution = """
    SELECT nationality_name, age, COUNT(*) AS num_players
    FROM fifa_players
    GROUP BY nationality_name, age
    ORDER BY nationality_name, age
"""
age_distribution_df = spark.sql(query_age_distribution).toPandas()
age_distribution_df.head()

In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(x='age', y='num_players', hue='nationality_name', data=age_distribution_df)
plt.title('Distribucion de Edades de Jugadores por Nacionalidad')
plt.xlabel('Edad')
plt.ylabel('Numero de Jugadores')
plt.xticks(rotation=45)
plt.legend(title='Nacionalidad', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Average Attributes by Age and Nationality

In [ ]:
query_avg_attributes = """
    SELECT nationality_name, age,
           AVG(pace) AS avg_pace,
           AVG(shooting) AS avg_shooting,
           AVG(passing) AS avg_passing,
           AVG(dribbling) AS avg_dribbling
    FROM fifa_players
    GROUP BY nationality_name, age
    ORDER BY nationality_name, age
"""
avg_attributes_df = spark.sql(query_avg_attributes).toPandas()
avg_attributes_df.head()

In [ ]:
plt.figure(figsize=(12, 8))
for nationality, data in avg_attributes_df.groupby('nationality_name'):
    plt.plot(data['age'], data['avg_pace'], marker='o', label=nationality)

plt.title('Comparacion de Atributos (Pace) por Edad y Nacionalidad')
plt.xlabel('Edad')
plt.ylabel('Promedio de Pace')
plt.xticks(rotation=45)
plt.legend(title='Nacionalidad', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Top Young Players by Potential

In [ ]:
query_young_players = """
    SELECT short_name, nationality_name, age, potential
    FROM fifa_players
    WHERE age < 21
    GROUP BY short_name, age, nationality_name, potential
    ORDER BY potential DESC
    LIMIT 10
"""
young_players_df = spark.sql(query_young_players).toPandas()
young_players_df.head()

In [ ]:
young_players_df['potential'] = pd.to_numeric(young_players_df['potential'], errors='coerce')
plt.figure(figsize=(10, 6))
sns.barplot(x='potential', y='short_name', hue='nationality_name', data=young_players_df)
plt.title('Top 10 de Jugadores Jovenes con Alto Potencial')
plt.xlabel('Potencial')
plt.ylabel('Nombre del Jugador')
plt.tight_layout()
plt.show()

## Average Attributes by Club Position

In [ ]:
query_position_attributes = """
    SELECT club_position,
           AVG(pace) AS avg_pace,
           AVG(shooting) AS avg_shooting,
           AVG(passing) AS avg_passing,
           AVG(dribbling) AS avg_dribbling
    FROM fifa_players
    GROUP BY club_position
"""
position_attributes_df = spark.sql(query_position_attributes).toPandas()
position_attributes_df.head()

In [ ]:
plt.figure(figsize=(12, 8))
position_attributes_df.set_index('club_position')[[
    'avg_pace', 'avg_shooting', 'avg_passing', 'avg_dribbling'
]].plot(kind='bar', stacked=False)
plt.title('Comparacion de Atributos por Posicion en el Club')
plt.xlabel('Posicion en el Club')
plt.ylabel('Promedio de Atributos')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Average Market Value by Nationality

In [ ]:
query_avg_value_by_nationality = """
    SELECT nationality_name,
           AVG(value_eur) AS avg_value_eur
    FROM fifa_players
    GROUP BY nationality_name
    ORDER BY avg_value_eur DESC
"""
avg_value_by_nationality_df = spark.sql(query_avg_value_by_nationality).toPandas()
avg_value_by_nationality_df.head()

In [ ]:
plt.figure(figsize=(20, 22))
sns.barplot(x='avg_value_eur', y='nationality_name', data=avg_value_by_nationality_df)
plt.title('Valor de Mercado Promedio por Nacionalidad')
plt.xlabel('Valor de Mercado Promedio (EUR)')
plt.ylabel('Nacionalidad')
plt.tight_layout()
plt.show()

In [ ]:
spark.stop()